# PicoGPT — Entraînement depuis zéro
Ce notebook entraîne un petit modèle GPT-2 custom sur un dataset de type instruction/réponse.

## 1. Imports

In [1]:
from transformers import (
    GPT2Config,
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments
)
from datasets import Dataset

c:\Users\nboud\Documents\GitHub-My-Projects\picogpt1-scratch-python\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Tokenizer

In [2]:
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

print(f"Vocab size : {len(tokenizer)}")
print(f"PAD token  : {tokenizer.pad_token!r}")

Vocab size : 50257
PAD token  : '<|endoftext|>'


## 3. Chargement du dataset
Format attendu dans `train.txt` : deux lignes par exemple (prompt + réponse), séparées par une ligne vide.

In [3]:
MAX_LEN = 256
MAX_EXAMPLES = 500

def build_example(prompt, completion):
    text = f"{prompt}\n{completion}{tokenizer.eos_token}"
    tokens = tokenizer(text, truncation=True, max_length=MAX_LEN)
    input_ids = tokens["input_ids"]

    labels = [-100] * len(input_ids)
    prefix_ids = tokenizer(f"{prompt}\n", truncation=True, max_length=MAX_LEN)["input_ids"]
    labels[len(prefix_ids):] = input_ids[len(prefix_ids):]

    return {
        "input_ids": input_ids,
        "attention_mask": tokens["attention_mask"],
        "labels": labels,
    }


def load_examples_from_file(path):
    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    blocks = [b.strip() for b in content.split("\n\n")]
    examples = []
    for block in blocks:
        if not block:
            continue
        lines = [line.strip() for line in block.splitlines() if line.strip()]
        if len(lines) < 2:
            continue
        examples.append(build_example(lines[0], lines[1]))
        if len(examples) >= MAX_EXAMPLES:
            break
    return examples


examples = load_examples_from_file("train.txt")
dataset = Dataset.from_list(examples)

print(f"{len(dataset)} exemples chargés")
print("Exemple 0 :", dataset[0])

500 exemples chargés
Exemple 0 : {'input_ids': [23318, 1115, 9040, 329, 10589, 5448, 13, 198, 16, 13, 47659, 257, 12974, 5496, 290, 787, 1654, 284, 2291, 6088, 286, 15921, 290, 13701, 13, 50256], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, -100, -100, -100, -100, -100, -100, -100, 16, 13, 47659, 257, 12974, 5496, 290, 787, 1654, 284, 2291, 6088, 286, 15921, 290, 13701, 13, 50256]}


## 4. Architecture du modèle

In [4]:
config = GPT2Config(
    vocab_size=len(tokenizer),
    n_positions=256,
    n_ctx=256,
    n_embd=128,
    n_layer=2,
    n_head=2,
)

model = GPT2LMHeadModel(config)
model.resize_token_embeddings(len(tokenizer))

total_params = sum(p.numel() for p in model.parameters())
print(f"Paramètres totaux : {total_params:,}")

Paramètres totaux : 6,862,464


## 5. Entraînement

In [5]:
args = TrainingArguments(
    output_dir="PicoGPT1-Scratch",
    per_device_train_batch_size=4,
    num_train_epochs=10,
    learning_rate=5e-4,
    logging_steps=50,
    save_steps=1000,
    save_total_limit=1,
    fp16=False,
    report_to="none",
    dataloader_num_workers=0,
)

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset,
    data_collator=collator,
)

trainer.train()

c:\Users\nboud\Documents\GitHub-My-Projects\picogpt1-scratch-python\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,9.510054
100,7.715101
150,7.113968
200,6.775078
250,6.655180
300,6.313286
350,6.323276
400,6.183776
450,6.060346
500,6.061331


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 36.12it/s]
c:\Users\nboud\Documents\GitHub-My-Projects\picogpt1-scratch-python\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 50.90it/s]


TrainOutput(global_step=1250, training_loss=5.961056457519531, metrics={'train_runtime': 144.1887, 'train_samples_per_second': 34.677, 'train_steps_per_second': 8.669, 'total_flos': 1073236070400.0, 'train_loss': 5.961056457519531, 'epoch': 10.0})

## 6. Sauvegarde du modèle

In [6]:
model.save_pretrained("PicoGPT1-Scratch")
tokenizer.save_pretrained("PicoGPT1-Scratch")
print("Modèle sauvegardé dans ./PicoGPT1-Scratch")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 44.12it/s]

Modèle sauvegardé dans ./PicoGPT1-Scratch
